# Augment and Dedupe Receipts (one-time migration)

Backfills missing `event_ts` / `event_date` columns on legacy Silver receipt
headers, lines, payments, and online-order headers. It also enforces the
business keys for receipt headers, receipt lines, and payments.

## Safety contract
- Exact duplicate headers are collapsed by `receipt_id_ext`; conflicting
  headers fail closed instead of choosing an arbitrary transaction.
- Exact duplicate lines and payments are collapsed by
  (`receipt_id_ext`, `line_num`) and `transaction_id`, respectively.
- Every candidate is materialized and validated in a run-scoped staging
  schema before promotion. Existing targets are backed up and restored if
  promotion or post-promotion validation fails.

The legacy receipt identifier format encodes the event time:
```
  RCP YYYYMMDDHHMM <store_id padded> <sequence>
  e.g. RCP 202412011318 01 25587
```
The timestamp is parsed from positions 4-15 and propagated from each validated
header to its lines and payments.

## Usage
Run **once** in Fabric against the `retail_lakehouse` Lakehouse, then refresh
the `retail_model` semantic model. The repair is idempotent.

In [ ]:
import os
import uuid

from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

In [ ]:
LAKEHOUSE_NAME = os.environ.get("LAKEHOUSE_NAME", "retail_lakehouse")
SILVER_DB = os.environ.get("SILVER_DB", "ag")
SILVER_STAGE_DB = os.environ.get("SILVER_STAGE_DB", f"{SILVER_DB}_stage")
RUN_TOKEN = uuid.uuid4().hex

RECEIPTS_TABLE = f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_receipts"
LINES_TABLE = f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_receipt_lines"
PAYMENTS_TABLE = f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_payments"
ONLINE_ORDERS_TABLE = (
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_online_order_headers"
)

spark.sql(
    f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{SILVER_STAGE_DB}"
)
print(
    f"Targeting {LAKEHOUSE_NAME}.{SILVER_DB}; "
    f"staging in {SILVER_STAGE_DB} for run {RUN_TOKEN}"
)

In [ ]:
def parse_event_ts_from_receipt_id(column_name):
    """Parse RCP{YYYYMMDDHHMM}{store}{sequence} identifiers."""
    ts_string = F.when(
        F.col(column_name).rlike("^RCP[0-9]{12}"),
        F.substring(F.col(column_name), 4, 12),
    )
    return F.to_timestamp(ts_string, "yyyyMMddHHmm")


def schema_signature(frame):
    return tuple(
        (field.name, field.dataType.simpleString())
        for field in frame.schema.fields
    )


def deduplicate_exact_rows(frame, key_columns, context):
    """Collapse exact replays and reject conflicting rows for a business key."""
    null_key = F.col(key_columns[0]).isNull()
    for column_name in key_columns[1:]:
        null_key = null_key | F.col(column_name).isNull()
    if frame.filter(null_key).limit(1).count():
        raise RuntimeError(f"{context} contains a null dedupe key: {key_columns}")

    comparable = (
        frame.drop("__index_level_0__")
        if "__index_level_0__" in frame.columns
        else frame
    )
    distinct_rows = comparable.dropDuplicates()
    conflicts = (
        distinct_rows
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .select(*key_columns)
        .limit(10)
        .collect()
    )
    if conflicts:
        samples = [row.asDict(recursive=True) for row in conflicts]
        raise RuntimeError(
            f"{context}: conflicting rows share dedupe key "
            f"{key_columns}: {samples}"
        )
    return distinct_rows.dropDuplicates(key_columns)


def validate_materialized_table(
    table_name,
    expected_count,
    expected_schema,
    context,
):
    materialized = spark.table(table_name)
    actual_count = materialized.count()
    actual_schema = schema_signature(materialized)
    if actual_count != expected_count or actual_schema != expected_schema:
        raise RuntimeError(
            f"{context} validation failed for {table_name}: "
            f"count {actual_count} != {expected_count} or "
            f"schema {actual_schema} != {expected_schema}"
        )


def stage_and_promote(table_frames):
    """Stage and validate every candidate before replacing any target."""
    staged = []
    for target_table, frame in table_frames:
        table_name = target_table.rsplit(".", 1)[-1]
        candidate_table = (
            f"{LAKEHOUSE_NAME}.{SILVER_STAGE_DB}."
            f"{RUN_TOKEN}__{table_name}__candidate"
        )
        expected_count = frame.count()
        expected_schema = schema_signature(frame)
        (
            frame.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(candidate_table)
        )
        validate_materialized_table(
            candidate_table,
            expected_count,
            expected_schema,
            "Staging",
        )
        staged.append(
            {
                "target": target_table,
                "candidate": candidate_table,
                "expected_count": expected_count,
                "expected_schema": expected_schema,
            }
        )

    promoted = []
    try:
        for item in staged:
            table_name = item["target"].rsplit(".", 1)[-1]
            backup_table = (
                f"{LAKEHOUSE_NAME}.{SILVER_STAGE_DB}."
                f"{RUN_TOKEN}__{table_name}__backup"
            )
            original = spark.table(item["target"])
            item["original_count"] = original.count()
            item["original_schema"] = schema_signature(original)
            item["backup"] = backup_table
            (
                original.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(backup_table)
            )
            validate_materialized_table(
                backup_table,
                item["original_count"],
                item["original_schema"],
                "Backup",
            )

        for item in staged:
            promoted.append(item)
            spark.sql(
                f"CREATE OR REPLACE TABLE {item['target']} "
                f"USING DELTA AS SELECT * FROM {item['candidate']}"
            )
            validate_materialized_table(
                item["target"],
                item["expected_count"],
                item["expected_schema"],
                "Promotion",
            )
    except Exception as exc:
        rollback_errors = []
        for item in reversed(promoted):
            try:
                spark.sql(
                    f"CREATE OR REPLACE TABLE {item['target']} "
                    f"USING DELTA AS SELECT * FROM {item['backup']}"
                )
                validate_materialized_table(
                    item["target"],
                    item["original_count"],
                    item["original_schema"],
                    "Rollback",
                )
            except Exception as rollback_exc:
                rollback_errors.append(
                    f"{item['target']}: {rollback_exc}"
                )
        preserved = [
            item.get("backup", item["candidate"])
            for item in staged
        ] + [item["candidate"] for item in staged]
        if rollback_errors:
            raise RuntimeError(
                f"Promotion failed: {exc}; rollback was incomplete. "
                f"Preserved staging: {preserved}; "
                f"rollback errors: {rollback_errors}"
            ) from exc
        raise RuntimeError(
            f"Promotion failed: {exc}; any changed targets were restored. "
            f"Preserved staging: {preserved}"
        ) from exc

    for item in staged:
        spark.sql(f"DROP TABLE IF EXISTS {item['candidate']}")
        spark.sql(f"DROP TABLE IF EXISTS {item['backup']}")
        print(
            f"Promoted {item['target']} "
            f"({item['expected_count']:,} rows)"
        )

## Step 1 — Augment + dedupe `fact_receipts`

In [ ]:
receipts = spark.read.table(RECEIPTS_TABLE)
print(f"Loaded {RECEIPTS_TABLE} ({receipts.count():,} rows)")
print("Columns before:", receipts.columns)

if "event_ts" not in receipts.columns:
    receipts = receipts.withColumn(
        "event_ts",
        parse_event_ts_from_receipt_id("receipt_id_ext"),
    )
else:
    receipts = receipts.withColumn(
        "event_ts",
        F.coalesce(
            F.col("event_ts"),
            parse_event_ts_from_receipt_id("receipt_id_ext"),
        ),
    )

if "event_date" not in receipts.columns:
    receipts = receipts.withColumn("event_date", F.to_date(F.col("event_ts")))
else:
    receipts = receipts.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.to_date(F.col("event_ts"))),
    )

null_dates = receipts.filter(F.col("event_date").isNull()).count()
print(f"Rows with null event_date after augmentation: {null_dates:,}")

In [ ]:
duplicate_receipt_keys = (
    receipts
    .groupBy("receipt_id_ext")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(
    "Duplicate receipt_id_ext values before exact-row dedupe: "
    f"{duplicate_receipt_keys:,}"
)

receipts_clean = deduplicate_exact_rows(
    receipts,
    ["receipt_id_ext"],
    "Receipt headers",
)
print(f"Receipt rows after validated dedupe: {receipts_clean.count():,}")

## Step 2 — Propagate `event_ts` / `event_date` to `fact_receipt_lines`

In [ ]:
lines = spark.read.table(LINES_TABLE)
print(f"Loaded {LINES_TABLE} ({lines.count():,} rows)")
print("Columns before:", lines.columns)

header_dates = receipts_clean.select(
    "receipt_id_ext",
    F.col("event_ts").alias("_hdr_event_ts"),
    F.col("event_date").alias("_hdr_event_date"),
)
lines_joined = lines.join(header_dates, on="receipt_id_ext", how="left")

if "event_ts" not in lines.columns:
    lines_aug = lines_joined.withColumn("event_ts", F.col("_hdr_event_ts"))
else:
    lines_aug = lines_joined.withColumn(
        "event_ts",
        F.coalesce(F.col("event_ts"), F.col("_hdr_event_ts")),
    )

if "event_date" not in lines.columns:
    lines_aug = lines_aug.withColumn("event_date", F.col("_hdr_event_date"))
else:
    lines_aug = lines_aug.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.col("_hdr_event_date")),
    )

lines_aug = lines_aug.drop("_hdr_event_ts", "_hdr_event_date")
lines_clean = deduplicate_exact_rows(
    lines_aug,
    ["receipt_id_ext", "line_num"],
    "Receipt lines",
)
missing_line_dates = lines_clean.filter(F.col("event_date").isNull()).count()
print(f"Rows missing event_date after join: {missing_line_dates:,}")
print(f"Receipt line rows after validated dedupe: {lines_clean.count():,}")

## Step 3 — Propagate `event_ts` / `event_date` to `fact_payments`

In [ ]:
payments = spark.read.table(PAYMENTS_TABLE)
print(f"Loaded {PAYMENTS_TABLE} ({payments.count():,} rows)")
print("Columns before:", payments.columns)

payments_joined = payments.join(
    header_dates,
    on="receipt_id_ext",
    how="left",
)

if "event_ts" not in payments.columns:
    payments_aug = payments_joined.withColumn(
        "event_ts",
        F.col("_hdr_event_ts"),
    )
else:
    payments_aug = payments_joined.withColumn(
        "event_ts",
        F.coalesce(F.col("event_ts"), F.col("_hdr_event_ts")),
    )

if "event_date" not in payments.columns:
    payments_aug = payments_aug.withColumn(
        "event_date",
        F.col("_hdr_event_date"),
    )
else:
    payments_aug = payments_aug.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.col("_hdr_event_date")),
    )

payments_aug = payments_aug.drop("_hdr_event_ts", "_hdr_event_date")
payments_clean = deduplicate_exact_rows(
    payments_aug,
    ["transaction_id"],
    "Payments",
)
missing_payment_dates = payments_clean.filter(
    F.col("receipt_id_ext").isNotNull() & F.col("event_date").isNull()
).count()
print(
    "In-store payments missing event_date after join: "
    f"{missing_payment_dates:,}"
)
print(f"Payment rows after validated dedupe: {payments_clean.count():,}")

stage_and_promote(
    [
        (RECEIPTS_TABLE, receipts_clean),
        (LINES_TABLE, lines_clean),
        (PAYMENTS_TABLE, payments_clean),
    ]
)

## Step 3b — Backfill `event_date` on `fact_online_order_headers`

Online order headers always carry `event_ts`, but legacy rows predate the
`event_date` column the semantic model now uses for its `dim_date`
relationship (Direct Lake requires date-to-date relationship columns).


In [ ]:
orders = spark.read.table(ONLINE_ORDERS_TABLE)
print(f"Loaded {ONLINE_ORDERS_TABLE} ({orders.count():,} rows)")
print("Columns before:", orders.columns)

if "event_date" not in orders.columns:
    orders_aug = orders.withColumn(
        "event_date",
        F.to_date(F.col("event_ts")),
    )
else:
    orders_aug = orders.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.to_date(F.col("event_ts"))),
    )

if "__index_level_0__" in orders_aug.columns:
    orders_aug = orders_aug.drop("__index_level_0__")

missing_orders = orders_aug.filter(F.col("event_date").isNull()).count()
print(f"Rows missing event_date after backfill: {missing_orders:,}")
stage_and_promote([(ONLINE_ORDERS_TABLE, orders_aug)])

## Step 3c — Normalize legacy PascalCase columns on promotions / foot traffic / store ops

Early datagen exports wrote PascalCase column names (`PromoCode`, `StoreID`,
`Count`, ...) while the v2 pipeline writes snake_case. Tables loaded from the
legacy exports therefore don't match the semantic model (snake_case per the
project convention), which breaks the Promotions / Foot Traffic / Store
Operations visuals. This step renames legacy columns (coalescing when both
spellings exist), backfills `event_ts` / `event_date`, and drops the pandas
index column.


In [ ]:
LEGACY_RENAMES = {
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_promotions": {
        "EventTS": "event_ts",
        "TraceId": "trace_id",
        "ReceiptId": "receipt_id_ext",
        "PromoCode": "promo_code",
        "DiscountAmount": "discount_amount",
        "DiscountCents": "discount_cents",
        "DiscountType": "discount_type",
        "ProductCount": "product_count",
        "ProductIds": "product_ids",
        "StoreID": "store_id",
        "CustomerID": "customer_id",
    },
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_foot_traffic": {
        "EventTS": "event_ts",
        "TraceId": "trace_id",
        "StoreID": "store_id",
        "SensorId": "sensor_id",
        "Zone": "zone",
        "Dwell": "dwell_seconds",
        "Count": "count",
    },
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_store_ops": {
        "EventTS": "event_ts",
        "TraceId": "trace_id",
        "StoreID": "store_id",
        "OperationTime": "operation_time",
        "OperationType": "operation_type",
    },
}

for table, renames in LEGACY_RENAMES.items():
    try:
        df = spark.read.table(table)
    except AnalysisException:
        print(f"Skipping {table}: table does not exist")
        continue

    print(f"\n=== {table} ===")
    print("Columns before:", df.columns)

    for legacy, snake in renames.items():
        if legacy not in df.columns:
            continue
        if snake in df.columns:
            # both spellings exist (legacy load + v2 streaming): coalesce into snake_case
            df = df.withColumn(snake, F.coalesce(F.col(snake), F.col(legacy))).drop(legacy)
        else:
            df = df.withColumnRenamed(legacy, snake)

    # store_ops legacy rows have operation_time but no event_ts
    if "event_ts" in df.columns and "operation_time" in df.columns:
        df = df.withColumn("event_ts", F.coalesce(F.col("event_ts"), F.col("operation_time")))
    elif "event_ts" not in df.columns and "operation_time" in df.columns:
        df = df.withColumn("event_ts", F.col("operation_time"))

    if "event_date" in df.columns:
        df = df.withColumn("event_date", F.coalesce(F.col("event_date"), F.to_date(F.col("event_ts"))))
    elif "event_ts" in df.columns:
        df = df.withColumn("event_date", F.to_date(F.col("event_ts")))

    if "__index_level_0__" in df.columns:
        df = df.drop("__index_level_0__")

    stage_and_promote([(table, df)])
    print("Columns after:", spark.read.table(table).columns)


## Step 4 — Verification

In [ ]:
for table in (
    RECEIPTS_TABLE,
    LINES_TABLE,
    PAYMENTS_TABLE,
    ONLINE_ORDERS_TABLE,
):
    frame = spark.read.table(table)
    print(f"\n=== {table} ===")
    print("Columns:", frame.columns)
    print("Rows:", frame.count())
    if "event_date" in frame.columns:
        frame.agg(
            F.min("event_date").alias("min_date"),
            F.max("event_date").alias("max_date"),
            F.sum(
                F.when(F.col("event_date").isNull(), 1).otherwise(0)
            ).alias("null_dates"),
        ).show()

dedupe_contracts = {
    RECEIPTS_TABLE: ["receipt_id_ext"],
    LINES_TABLE: ["receipt_id_ext", "line_num"],
    PAYMENTS_TABLE: ["transaction_id"],
}
for table, key_columns in dedupe_contracts.items():
    duplicate_keys = (
        spark.read.table(table)
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    print(f"Duplicate keys remaining in {table}: {duplicate_keys:,}")
    assert duplicate_keys == 0, (
        f"Duplicate keys remain in {table}: {key_columns}"
    )

print(
    "\nReady. Refresh the retail_model semantic model in Fabric "
    "to pick up the repaired tables."
)